---

Created for [Pricing and Hedging Derivative Securities: Theory and Methods](https://book.derivative-securities.org/)

Authored by
- Kerry Back, Rice University
- Hong Liu, Washington University in St. Louis
- Mark Loewenstein, University of Maryland
 
---

<a target="_blank" href="https://colab.research.google.com/github/math-finance-book/book-code/blob/main/03_Trees.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:

import plotly
from IPython.display import display, HTML

plotly.offline.init_notebook_mode(connected=True)
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

In [ ]:

import numpy as np
def binomial_american_detailed(S0, K, r, sigma, T, N, option_type='put'):
    """
    Price American option using binomial tree with detailed backward induction
    
    This function shows exactly how backward induction works step by step
    """
    # Step 1: Set up tree parameters
    dt = T / N                          # Time per step
    u = np.exp(sigma * np.sqrt(dt))     # Up factor
    d = 1 / u                           # Down factor (ensures recombining tree)
    p = (np.exp(r * dt) - d) / (u - d)  # Risk-neutral probability
    disc = np.exp(-r * dt)              # Discount factor
    
    print(f"Tree parameters: u={u:.4f}, d={d:.4f}, p={p:.4f}")
    
    # Step 2: Initialize option values at maturity (time N)
    # At maturity, we have N+1 possible stock prices
    V = np.zeros(N+1)  # Option values
    S = np.zeros(N+1)  # Stock prices
    
    for j in range(N+1):
        S[j] = S0 * (u**j) * (d**(N-j))  # Stock price at node j
        if option_type == 'call':
            V[j] = max(0, S[j] - K)      # Call payoff
        else:
            V[j] = max(0, K - S[j])      # Put payoff
    
    print(f"\nAt maturity (time {N}):")
    print(f"Stock prices: {[f'{s:.2f}' for s in S]}")
    print(f"Option values: {[f'{v:.4f}' for v in V]}")
    
    # Step 3: Backward induction through the tree
    for i in range(N-1, -1, -1):  # Work backwards from time N-1 to 0
        print(f"\nTime step {i}:")
        
        # At time i, we have i+1 nodes
        new_V = np.zeros(i+1)
        new_S = np.zeros(i+1)
        
        for j in range(i+1):
            # Stock price at this node
            new_S[j] = S0 * (u**j) * (d**(i-j))
            
            # Continuation value (discounted expected value)
            continuation = disc * (p * V[j+1] + (1-p) * V[j])
            
            # Immediate exercise value
            if option_type == 'call':
                exercise = max(0, new_S[j] - K)
            else:
                exercise = max(0, K - new_S[j])
            
            # American option: take maximum of continuation and exercise
            new_V[j] = max(continuation, exercise)
            
            print(f"  Node {j}: S={new_S[j]:.2f}, Cont={continuation:.4f}, "
                  f"Exercise={exercise:.4f}, Value={new_V[j]:.4f}")
        
        # Update for next iteration
        V = new_V.copy()
        S = new_S.copy()
    
    return V[0]

# Example with small tree to see the process
print("Detailed backward induction for 2-step American put:")
S0, K, r, sigma, T, N = 100, 105, 0.05, 0.2, 1.0, 2
put_value = binomial_american_detailed(S0, K, r, sigma, T, N, 'put')
print(f"\nFinal American put value: {put_value:.4f}")

In [ ]:

import pydot
from IPython.display import Image, display
import numpy as np

# Use the same parameters as the detailed example above
S0 = 100  # Initial stock price
K = 105   # Strike price  
r = 0.05  # Risk-free rate
sigma = 0.2  # Volatility
T = 1.0   # Time to maturity
N = 2     # Number of periods

# Calculate tree parameters
dt = T / N
u = np.exp(sigma * np.sqrt(dt))
d = 1 / u
p = (np.exp(r * dt) - d) / (u - d)
disc = np.exp(-r * dt)

# Calculate all stock prices and option values using same logic as detailed function
V = np.zeros(N+1)  # Option values
S = np.zeros(N+1)  # Stock prices

# Time 2 (maturity) - put option payoffs
for j in range(N+1):
    S[j] = S0 * (u**j) * (d**(N-j))
    V[j] = max(0, K - S[j])  # Put payoff

S_20, S_21, S_22 = S[0], S[1], S[2]
V_20, V_21, V_22 = V[0], V[1], V[2]

# Time 1 - backward induction  
V_new = np.zeros(2)
S_new = np.zeros(2)
for j in range(2):
    S_new[j] = S0 * (u**j) * (d**(1-j))
    continuation = disc * (p * V[j+1] + (1-p) * V[j])
    exercise = max(0, K - S_new[j])
    V_new[j] = max(continuation, exercise)

S_10, S_11 = S_new[0], S_new[1]
V_10, V_11 = V_new[0], V_new[1]
V = V_new.copy()

# Time 0 - final backward induction
S_00 = S0
continuation = disc * (p * V[1] + (1-p) * V[0])
exercise = max(0, K - S_00)
V_00 = max(continuation, exercise)

# Create the graph
graph = pydot.Dot(graph_type='digraph', rankdir='LR', bgcolor='white')
graph.set_node_defaults(shape='circle', style='filled', fillcolor='lightblue', 
                       fontname='Arial', fontsize='9')
graph.set_edge_defaults(fontname='Arial', fontsize='8')

# Time 0
graph.add_node(pydot.Node('t0', 
                         label=f't=0\\nS={S_00:.2f}\\nV={V_00:.4f}',
                         fillcolor='lightgreen'))

# Time 1
graph.add_node(pydot.Node('t1_u', 
                         label=f't=1 Up\\nS={S_11:.2f}\\nV={V_11:.4f}',
                         fillcolor='lightyellow'))
graph.add_node(pydot.Node('t1_d', 
                         label=f't=1 Down\\nS={S_10:.2f}\\nV={V_10:.4f}',
                         fillcolor='lightyellow'))

# Time 2 (maturity)
graph.add_node(pydot.Node('t2_uu', 
                         label=f't=2 UU\\nS={S_22:.2f}\\nV={V_22:.4f}',
                         fillcolor='lightcoral'))
graph.add_node(pydot.Node('t2_ud', 
                         label=f't=2 UD\\nS={S_21:.2f}\\nV={V_21:.4f}',
                         fillcolor='lightcoral'))
graph.add_node(pydot.Node('t2_dd', 
                         label=f't=2 DD\\nS={S_20:.2f}\\nV={V_20:.4f}',
                         fillcolor='lightcoral'))

# Add edges
graph.add_edge(pydot.Edge('t0', 't1_u', label=f'p={p:.3f}'))
graph.add_edge(pydot.Edge('t0', 't1_d', label=f'1-p={1-p:.3f}'))

graph.add_edge(pydot.Edge('t1_u', 't2_uu', label=f'p={p:.3f}'))
graph.add_edge(pydot.Edge('t1_u', 't2_ud', label=f'1-p={1-p:.3f}'))
graph.add_edge(pydot.Edge('t1_d', 't2_ud', label=f'p={p:.3f}'))
graph.add_edge(pydot.Edge('t1_d', 't2_dd', label=f'1-p={1-p:.3f}'))

# Save and display
graph.write_png('two_step_tree.png')
display(Image('two_step_tree.png'))

print(f"Tree parameters matching the numerical example:")
print(f"u = {u:.4f}, d = {d:.4f}")
print(f"Risk-neutral probability p = {p:.4f}")
print(f"\nThis tree shows the exact same values computed in the numerical output above.")

In [ ]:

def binomial_american_fast(S0, K, r, sigma, T, N, option_type='put'):
    """
    Efficient American option pricing using backward induction
    
    This version uses vectorized operations for speed
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    
    # Initialize option values at maturity
    # Use vectorized operations for efficiency
    j_values = np.arange(N+1)
    S_final = S0 * (u**j_values) * (d**(N-j_values))
    
    if option_type == 'call':
        V = np.maximum(S_final - K, 0)
    else:
        V = np.maximum(K - S_final, 0)
    
    # Backward induction
    for i in range(N-1, -1, -1):
        j_values = np.arange(i+1)
        S_current = S0 * (u**j_values) * (d**(i-j_values))
        
        # Continuation values (vectorized)
        V_new = disc * (p * V[1:i+2] + (1-p) * V[0:i+1])
        
        # Exercise values
        if option_type == 'call':
            exercise = np.maximum(S_current - K, 0)
        else:
            exercise = np.maximum(K - S_current, 0)
        
        # Take maximum (American feature)
        V = np.maximum(V_new, exercise)
    
    return V[0]

# Examples
print("\nAmerican option values:")
put_american = binomial_american_fast(100, 105, 0.05, 0.2, 1, 100, 'put')
call_american = binomial_american_fast(100, 105, 0.05, 0.2, 1, 100, 'call')
print(f"American put (K=105): {put_american:.4f}")
print(f"American call (K=105): {call_american:.4f}")